In [ ]:
#==================================
# 地形指标提取脚本 - 坐标兼容增强版
# 作者：Gemini
# 备注：能用，好用，缩短10倍时间，但不知道为什么，结果跟调用arcpy差0.1左右
#
#==================================

import arcpy
import numpy as np
import csv
import time
import os
import math

# ==================== 参数配置 ====================
dem_path = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\Extract_buff1" 
slope_path = r"D:\alb_prj_Bigdata\Bigdata_alban.gdb\Slope_5km"
aspect_path = r"D:\alb_prj_Bigdata\Bigdata_alban.gdb\Aspect_5km"
output_csv = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\output\test.csv"
buffer_layer = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\Buffer_5km"

arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

# ==================== 初始化与坐标同步 ====================
print("正在同步坐标参考系统...")
desc_raster = arcpy.Describe(slope_path)
raster_sr = desc_raster.spatialReference # 获取栅格的坐标系
cell_size = desc_raster.meanCellWidth
pixel_radius = int(5000 / cell_size)

# 确保 POI 也在内存中按栅格坐标系读取
# 使用投影后的几何对象进行计算
print(f"当前栅格分辨率: {cell_size}，缓冲区像素半径: {pixel_radius}")

# ==================== 核心处理循环 ====================
results = []
header = ["OID", "Mean_Slope", "Northness", "Eastness", "Solar_Radiation_Index", "Terrain_Heterogeneity"]

start_time = time.time()
processed_count = 0
error_count = 0

# 在 SearchCursor 中明确请求投影到栅格坐标系的几何体
with arcpy.da.SearchCursor(buffer_layer, ["SHAPE@", "OID@"], spatial_reference=raster_sr) as cursor:
    for row in cursor:
        # 此时 buffer_geom 的坐标已自动转换至与栅格一致
        buffer_geom = row[0]
        oid = row[1]
        
        try:
            # 锁定 Extent
            ext = buffer_geom.extent
            arcpy.env.extent = ext
            
            # 明确指定读取参数
            cols = int(math.ceil((ext.XMax - ext.XMin) / cell_size)) + 2
            rows = int(math.ceil((ext.YMax - ext.YMin) / cell_size)) + 2

            # 读取局部数组
            slice_slope = arcpy.RasterToNumPyArray(slope_path, 
                                                   lower_left_corner=ext.lowerLeft,
                                                   ncols=cols, 
                                                   nrows=rows,
                                                   nodata_to_value=np.nan)
            
            # 如果数组全为空，打印坐标调试信息
            if np.isnan(slice_slope).all():
                # 打印第一个失败点的坐标，方便你在 ArcMap/Pro 中定位检查
                print(f"OID {oid} 提取数据全为 NaN。坐标范围: X[{ext.XMin:.1f}-{ext.XMax:.1f}]")
                error_count += 1
                continue

            slice_aspect = arcpy.RasterToNumPyArray(aspect_path, 
                                                    lower_left_corner=ext.lowerLeft,
                                                    ncols=cols, 
                                                    nrows=rows,
                                                    nodata_to_value=np.nan)
            
            # 构建局部掩膜
            h, w = slice_slope.shape
            y_idx, x_idx = np.ogrid[-h//2 : h - h//2, -w//2 : w - w//2]
            current_mask = (x_idx**2 + y_idx**2 <= pixel_radius**2)
            
            if current_mask.shape != slice_slope.shape:
                current_mask = current_mask[:h, :w]

            # 提取有效值
            mask_final = np.logical_and(current_mask, ~np.isnan(slice_slope))
            v_slope = slice_slope[mask_final]
            v_aspect = slice_aspect[mask_final]

            if v_slope.size == 0:
                continue

            # 指标计算
            m_slope = np.nanmean(v_slope)
            std_slope = np.nanstd(v_slope)
            aspect_rad = v_aspect * (np.pi / 180.0)
            m_north = np.nanmean(np.cos(aspect_rad))
            m_east = np.nanmean(np.sin(aspect_rad))
            slope_rad = v_slope * (np.pi / 180.0)
            m_psr = np.nanmean(np.cos(slope_rad) + (np.sin(slope_rad) * np.cos(aspect_rad)))
            
            results.append([oid, m_slope, m_north, m_east, m_psr, std_slope])
            
            processed_count += 1
            print(f"已处理 {processed_count} 个点...")

        except Exception as e:
            error_count += 1
            print(f"OID {oid} 处理异常: {str(e)}")

# ==================== 输出 ====================
arcpy.env.extent = None
with open(output_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(results)

print(f"任务结束。成功: {processed_count}, 失败: {error_count}。耗时: {(time.time()-start_time)/60:.2f}min")

正在同步坐标参考系统...
当前栅格分辨率: 1.0，缓冲区像素半径: 5000
已处理 1 个点...
已处理 2 个点...
已处理 3 个点...
已处理 4 个点...
已处理 5 个点...
已处理 6 个点...
已处理 7 个点...
已处理 8 个点...
已处理 9 个点...
已处理 10 个点...
已处理 11 个点...
已处理 12 个点...
已处理 13 个点...
已处理 14 个点...
已处理 15 个点...
已处理 16 个点...
已处理 17 个点...
已处理 18 个点...
已处理 19 个点...
已处理 20 个点...
已处理 21 个点...
已处理 22 个点...
已处理 23 个点...
已处理 24 个点...
已处理 25 个点...
已处理 26 个点...
已处理 27 个点...
已处理 28 个点...
已处理 29 个点...
已处理 30 个点...
已处理 31 个点...
已处理 32 个点...
已处理 33 个点...
已处理 34 个点...
已处理 35 个点...
已处理 36 个点...
已处理 37 个点...
已处理 38 个点...
已处理 39 个点...
已处理 40 个点...
已处理 41 个点...
已处理 42 个点...
已处理 43 个点...
已处理 44 个点...
已处理 45 个点...
已处理 46 个点...
已处理 47 个点...
已处理 48 个点...
已处理 49 个点...
已处理 50 个点...
已处理 51 个点...
已处理 52 个点...
已处理 53 个点...
已处理 54 个点...
已处理 55 个点...
已处理 56 个点...
已处理 57 个点...
已处理 58 个点...
已处理 59 个点...
已处理 60 个点...
已处理 61 个点...
已处理 62 个点...
已处理 63 个点...
已处理 64 个点...
已处理 65 个点...
已处理 66 个点...
已处理 67 个点...
已处理 68 个点...
已处理 69 个点...
已处理 70 个点...
已处理 71 个点...
已处理 72 个点...
已处理 73 个点...
已处理 74 个点...
已处理 75